In [1]:
import sys
sys.path.append("..")

In [2]:
import tqdm
import os
import time
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import cvxpy as cp
from copy import deepcopy

from src.data import *
from src.model import *
from src.recourse import *
from src.utils import *

warnings.filterwarnings('ignore')

In [3]:
def append_result(d, algorithm, seed, alpha, lamb, i, x_0, theta_0, x_r, theta_r=None):
    d["algorithm"].append(algorithm)
    d["seed"].append(seed)
    d["alpha"].append(alpha)
    d["lambda"].append(lamb)
    d["i"].append(i)
    d["x_0"].append(x_0.round(4))
    d["x_r"].append(x_r.round(4))
    d["theta_0"].append(theta_0.round(4))

def append_result_time(d_time, index, e_time):
    d_time["i"].append(index)
    d_time["e_time"].append(np.float64(e_time).round(5))

In [4]:
def getLearningRate(dataset_name, alpha, lamb):
    lr0 = None

    if alpha == 0.1:
        if dataset_name == 'german':
            if lamb <= 0.001:
                lr0 = 40
            elif lamb <= 0.005:
                lr0 = 30
            elif lamb <= 0.01:
                lr0 = 25
            elif lamb <= 0.05:
                lr0 = 15
            elif lamb <= 0.1:
                lr0 = 10
            elif lamb <= 0.3:
                lr0 = 5
            elif lamb <= 0.5:
                lr0 = 0.5
            elif lamb <= 1:
                lr0 = 0.1
            else:
                lr0 = 0.05

        elif dataset_name == 'sba':
            if lamb <= 0.001:
                lr0 = 10
            elif lamb <= 0.01:
                lr0 = 7.5
            elif lamb <= 0.1:
                lr0 = 5
            elif lamb <= 0.5:
                lr0 = 2.5
            elif lamb <= 2.5:
                lr0 = 1
            elif lamb <= 3.5:
                lr0 = 0.5
            else:
                lr0 = 25

        elif dataset_name == 'income':
            if lamb <= 0.01:
                lr0 = 8
            elif lamb <= 0.1:
                lr0 = 6
            elif lamb <= 0.5:
                lr0 = 4
            else:
                lr0 = 1 

    elif alpha == 0.5:
        if dataset_name == 'german':
            if lamb <= 0.001:
                lr0 = 40
            elif lamb <= 0.005:
                lr0 = 30
            elif lamb <= 0.01:
                lr0 = 25
            elif lamb <= 0.05:
                lr0 = 15
            elif lamb <= 0.1:
                lr0 = 10
            elif lamb <= 0.3:
                lr0 = 9
            elif lamb <= 0.5:
                lr0 = 7.5
            elif lamb <= 1:
                lr0 = 5
            else:
                lr0 = 0.05
        
        elif dataset_name == 'sba':
            if lamb <= 0.001:
                lr0 = 12.5
            elif lamb <= 0.01:
                lr0 = 11
            elif lamb <= 0.1:
                lr0 = 10
            elif lamb <= 0.5:
                lr0 = 7.5
            elif lamb <= 2.5:
                lr0 = 5
            elif lamb <= 3.5:
                lr0 = 2.5
            else:
                lr0 = 1.0         

        elif dataset_name == 'income':
            if lamb <= 0.01:
                lr0 = 14
            elif lamb <= 0.1:
                lr0 = 11
            elif lamb <= 0.5:
                lr0 = 9
            else:
                lr0 = 2.5   
    
    return lr0

In [5]:
def recourse_runner(seed: int, X: np.ndarray, recourse: Recourse, params: dict, dataset: Dataset):
    alpha = params['alpha']
    lamb = params['lamb']
    lr = getLearningRate(dataset.name, alpha, lamb)
    if recourse.name == "L1PSD":
        recourse.set_learning_rate(lr)
        print(recourse.lr)        
    recourse_i = params['recourse_index']
    
    f_name = f'../results/recourse/lr_{dataset.name}_{recourse.name}_{lamb}_{alpha}_{seed}.pkl'
    f_name_time = f'../results/recourse_time/lr_{dataset.name}_{recourse.name}_{lamb}_{alpha}_{seed}_time.pkl'
    
    results = {'algorithm': [], 'seed': [], 'alpha': [], 'lambda': [], 'i': [], 'x_0': [], 'x_r': [], 'theta_0': []}
    results_time = {'i': [], "e_time": []}

    weights_0, bias_0 = recourse.weights, recourse.bias
    theta_0 = np.hstack((weights_0, bias_0))
    if recourse.name == "ROAR":
        print(weights_0, bias_0, theta_0)
    n = len(X)

    for i in tqdm.trange(n, desc=f'[{recourse.name}] [alpha={alpha}] [lambda={lamb}]', colour='#0091ff'):
        x_0 = X[i]
        
        start_time = time.perf_counter()
        x_r = recourse.get_recourse(x_0)
        end_time = time.perf_counter()
        elapsed_time = end_time - start_time
        
        append_result(results, recourse.name, seed, alpha, lamb, recourse_i[i], x_0, theta_0, x_r)
        append_result_time(results_time, recourse_i[i], elapsed_time)

    df_results = pd.DataFrame(results)
    df_results_time = pd.DataFrame(results_time)

    if params['append_results'] and os.path.exists(f_name):
        df_tmp = pd.read_pickle(f_name)
        df_results = pd.concat((df_tmp, df_results), axis=0).sort_values(['i'], ignore_index=True)
    if params['append_results'] and params['save_time'] and os.path.exists(f_name_time):
        df_time_tmp = pd.read_pickle(f_name_time)
        df_results_time = pd.concat((df_time_tmp, df_results_time), axis=0).sort_values(['i'], ignore_index=True)        

    if params["save_results"]:
        print(f'[{recourse.name}] Saving results for {dataset.name} run {seed}')
        df_results.to_pickle(f_name)
    if params["save_time"]:
        print(f'[{recourse.name}] Saving time results for {dataset.name} run {seed}')
        df_results_time.to_pickle(f_name_time)
    
    return df_results

In [6]:
def run_experiment(dataset: Dataset, recourse_fns: List[Recourse], params: dict, results: List):
    alpha = params['alpha']
    lamb = params['lamb']
    
    for seed in params['seeds']:
        train_data, test_data = dataset.get_data(seed)
        X_train, y_train = train_data
        X_test, y_test = test_data
        
        base_model = LR()
        base_model.train(X_train.values, y_train.values)
        
        weights_0 = base_model.model.coef_[0]
        bias_0 = base_model.model.intercept_
        
        recourse_needed_X_train = recourse_needed(base_model.predict, X_train.values)
        recourse_needed_X_test = recourse_needed(base_model.predict, X_test.values)
        
        for recourse_fn in recourse_fns:
            recourse_needed_X_test_idx = np.arange(recourse_needed_X_test.shape[0])

            recourse = recourse_fn(weights=weights_0, bias=bias_0, alpha=alpha, lamb=lamb)
            if params["lamb"] is None:
                params['lamb'] = recourse.choose_lambda(recourse_needed_X_train, base_model.predict, X_train.values)
                recourse.lamb = params['lamb']

            if params['first_n']:
                recourse_needed_X_test_idx = recourse_needed_X_test_idx[:params['first_n_value']]
            
            if params['append_results']:
                f_name = f"../results/recourse/lr_{dataset.name}_{recourse.name}_{lamb}_{alpha}_{seed}.pkl"
                if os.path.exists(f_name):
                    df_tmp = pd.read_pickle(f_name)
                    recourse_needed_X_test_idx = np.setdiff1d(recourse_needed_X_test_idx, df_tmp['i'].to_numpy())
                else:
                    print(f"{f_name} does not exist. The recourses will be created in a new file")

                if recourse_needed_X_test_idx.size == 0:
                    print(f"{f_name} already has all the recourses. Skipping")
                    continue
            
            if params['subsample']:
                rng = np.random.default_rng(seed=seed)
                size_N = int(np.rint(params['subsample_size'] * recourse_needed_X_test.shape[0]))
                if recourse_needed_X_test_idx.shape[0] < size_N:
                    size_N = recourse_needed_X_test_idx.shape[0]
                recourse_needed_X_test_idx = rng.choice(recourse_needed_X_test_idx, size=size_N, replace=False)

            params['recourse_index'] = recourse_needed_X_test_idx.copy()
            recourse_needed_X_test = recourse_needed_X_test[recourse_needed_X_test_idx]
                
            df_results = recourse_runner(seed, recourse_needed_X_test, recourse, params, dataset)
            results.append(df_results)

In [9]:
alphas = [0.5] # <------------------------
lambdas = [0.1, 0.01, 0.5] # <------------------------

torch.manual_seed(0)

for lamb in lambdas:
    for alpha in alphas:

        d_results = {}
        params = {}
        params['alpha'] = alpha # float, None
        params['lamb'] = lamb
        params['seeds'] = range(2)
        params['append_results'] = False
        params['save_results'] = True
        params['save_time'] = True
        params['subsample'] = False
        params['subsample_size'] = 0.5
        params['first_n'] = True
        params['first_n_value'] = 16

        datasets = [IncomeDataset()] # <------------------------
        recourse_fns = [ROARLInf, ROARL1] # <------------------------

        for dataset in datasets:
            results = []
            print(f'Running {dataset.name} data...')
            run_experiment(dataset, recourse_fns, params, results)
            
            d_results[dataset.name] = pd.concat(results)
            print(f'Finished {dataset.name}\n')

Running income data...


[ROARLInf] [alpha=0.5] [lambda=0.1]: 100%|██████████| 16/16 [01:02<00:00,  3.92s/it]


[ROARLInf] Saving results for income run 0
[ROARLInf] Saving time results for income run 0


[ROARL1] [alpha=0.5] [lambda=0.1]: 100%|██████████| 16/16 [00:09<00:00,  1.74it/s]


[ROARL1] Saving results for income run 0
[ROARL1] Saving time results for income run 0


...

[ROARL1] Saving results for income run 0
[ROARL1] Saving time results for income run 0


[ROARLInf] [alpha=0.5] [lambda=0.5]: 100%|██████████| 16/16 [00:43<00:00,  2.70s/it]


[ROARLInf] Saving results for income run 1
[ROARLInf] Saving time results for income run 1


[ROARL1] [alpha=0.5] [lambda=0.5]: 100%|██████████| 16/16 [00:03<00:00,  4.85it/s]

[ROARL1] Saving results for income run 1
[ROARL1] Saving time results for income run 1
Finished income

